# MRI and Segmentation Mask Visualization

Renders axial/coronal/sagittal overlays of the four MRI modalities with ground-truth and predicted segmentation masks for qualitative inspection.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import copy
from scipy.stats import wilcoxon, mannwhitneyu
import pickle as pkl
from os import listdir
import os

import nibabel as nib

import seaborn as sns
import imageio
from skimage.transform import resize
from skimage.util import montage

import pydicom as pdm
import nilearn as nl
import nilearn.plotting as nlplt
import h5py

from matplotlib import cm
import matplotlib.animation as anim
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec


In [ ]:
def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET])
    
    return mask 

def get_tumor_slices(mri_mask):
    """
    Extracts slices from a 3D MRI mask where the tumor exists.

    Parameters:
    mri_mask (numpy.ndarray): A 3D NumPy array representing the MRI mask.

    Returns:
    list of numpy.ndarray: A list of 2D slices containing the tumor.
    """
    tumor_slices = []

    # Iterate through each slice
    for i in range(mri_mask.shape[2]):
        _slice = mri_mask[:, :, i]
        
        # Check if the slice contains tumor (non-zero values)
        if np.any(_slice):
            tumor_slices.append(i)

    return tumor_slices

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = 'BraTS-GLI-' + patient_id + '/' + 'BraTS-GLI-' + patient_id
        result_loc = '../Results/Result/Vanilla_Unet/BraTS-GLI-' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz', '-seg..npz']

    flair_filename = baseloc + pefix + suffixs[0]
    flair_img_f = nib.load(flair_filename)
    flair_img = np.asarray(flair_img_f.dataobj)

    t2_filename = baseloc + pefix + suffixs[1]
    t2_img_f = nib.load(t2_filename)
    t2_img = np.asarray(t2_img_f.dataobj)

    t1_filename = baseloc + pefix + suffixs[2]
    t1_img_f = nib.load(t1_filename)
    t1_img = np.asarray(t1_img_f.dataobj)

    t1ce_filename = baseloc + pefix + suffixs[3]
    t1ce_img_f = nib.load(t1ce_filename)
    t1ce_img = np.asarray(t1ce_img_f.dataobj)
    
    mask_filename = baseloc + pefix + suffixs[4]
    mask_img_f = nib.load(mask_filename)
    mask_img = np.asarray(mask_img_f.dataobj)
    
    try:
        output_filename = result_loc + suffixs[4]
        output_img_f = nib.load(output_filename)
        output_img = np.asarray(output_img_f.dataobj)

    except:
        print('Not in Test Set')
        return flair_img, t2_img, t1_img, t1ce_img, mask_img, 0, 0
    
    return flair_img, t2_img, t1_img, t1ce_img, mask_img, output_img#, probablity_img

def print_image_all(sample_img1, sample_img2, sample_img3, sample_img4, sample_mask, _slice):
    
    fig = plt.figure(figsize=(20, 30))

    gs = gridspec.GridSpec(nrows=4, ncols=2)

    #  Varying density along a streamline
    ax0 = fig.add_subplot(gs[0, 0])
    flair = ax0.imshow(sample_img1[:,:,_slice], cmap='gray')
    ax0.set_title("FLAIR", fontsize=18, weight='bold', y=-0.2)
    fig.colorbar(flair)

    #  Varying density along a streamline
    ax1 = fig.add_subplot(gs[0, 1])
    t1 = ax1.imshow(sample_img2[:,:,_slice], cmap='gray')
    ax1.set_title("T2", fontsize=18, weight='bold', y=-0.2)
    fig.colorbar(t1)

    #  Varying density along a streamline
    ax2 = fig.add_subplot(gs[1, 0])
    t2 = ax2.imshow(sample_img3[:,:,_slice], cmap='gray')
    ax2.set_title("T1", fontsize=18, weight='bold', y=-0.2)
    fig.colorbar(t2)

    #  Varying density along a streamline
    ax3 = fig.add_subplot(gs[1, 1])
    t1ce = ax3.imshow(sample_img4[:,:,_slice], cmap='gray')
    ax3.set_title("T1 contrast", fontsize=18, weight='bold', y=-0.2)
    fig.colorbar(t1ce)

    #  Varying density along a streamline
    ax4 = fig.add_subplot(gs[2, 0])
    
    masks = preprocess_mask_labels(sample_mask)
    
    mask_WT, mask_TC, mask_ET = masks[0], masks[1], masks[2]

    #ax4.imshow(np.ma.masked_where(mask_WT[:,:,65]== False,  mask_WT[:,:,65]), cmap='summer', alpha=0.6)
    l1 = ax4.imshow(mask_WT[:,:,_slice], cmap='gray',)
    l2 = ax4.imshow(np.ma.masked_where(mask_TC[:,:,_slice]== False,  mask_TC[:,:,_slice]), cmap='Dark2', alpha=0.6)
    l3 = ax4.imshow(np.ma.masked_where(mask_ET[:,:,_slice] == False, mask_ET[:,:,_slice]), cmap='winter', alpha=0.6)

    ax4.set_title("", fontsize=20, weight='bold', y=-0.1)

    _ = [ax.set_axis_off() for ax in [ax0,ax1,ax2,ax3, ax4]]

    colors = [im.cmap(im.norm(1)) for im in [l1,l2, l3]]
    labels = ['Peritumoral Edema ', 'Non-Enhancing tumor core', 'GD-enhancing tumor']
    patches = [ mpatches.Patch(color=colors[i], label=f"{labels[i]}") for i in range(len(labels))]
    # put those patched as legend-handles into the legend
    plt.legend(handles=patches, bbox_to_anchor=(1.1, 0.65), loc=2, borderaxespad=0.4,fontsize = 'xx-large',
               title='Mask Labels', title_fontsize=18, edgecolor="black",  facecolor='#c5c6c7')
    
def overlay_mask(mri_slice, mask_slice):
    # Assuming the tumor regions are labeled as 1, 2, 3 in the mask
    colors = {1: 'red_transparent', 2: 'green_transparent', 3: 'blue_transparent'}  # Assign colors to each tumor region
    masked_image = np.ma.masked_where(mask_slice == 0, mask_slice)
#     fig = plt.figure(figsize=(8, 8))
    plt.imshow(mri_slice, cmap='gray')  # Display the MRI slice in grayscale
    for label, color in colors.items():
        plt.imshow(np.ma.masked_where(masked_image != label, masked_image), cmap=color, alpha=0.5)
    
    plt.axis('off')
    plt.show()


In [ ]:
dataset = 'Brats2023'
radius = 1
data_path = '../input/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
result_path = '../Results/Analysis_Results/probability/GLI-Image_intensity_tumor_vs_non_tumor.pkl'
patient_ids = [f for f in listdir(data_path) if not os.path.isfile(os.path.join(data_path, f))]


In [ ]:
patient_id = '00008-000'
flair_img, t2_img, t1_img, t1ce_img, mask_img, output_img = read_MRI(dataset, patient_id)


In [ ]:
_slice = 65
print_image_all(flair_img, t2_img, t1_img, t1ce_img, mask_img, _slice)


In [ ]:
slice_index = 110
overlay_mask(t2_img[:,:,slice_index], output_img[:,:,slice_index])
